# Training and Validation for Oxidation Potential (Chemprop CLI)

The experimental-only retrain is `model53`; its metrics and manifest are in `chemprop_model_output/model53`. Saved cell outputs below were generated by the older model52 run and update when rerun.


## 1. Configuration


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

try:
    import pubchempy as pcp
except Exception:
    pcp = None

# Run controls
RUN_TRAIN = False
RUN_PREDICT = True
RUN_FIGURES = True

# Core config
PROPERTY_CODE = 'OP'
TARGET_COLUMN = 'OP'
SMILES_COLUMN = 'SMILES'
SPLIT_COLUMN = 'split'

BASE_OUTPUT_DIR = Path('./chemprop_model_output')
TRAIN_DIR = Path('./Datasets/trainable_data')
TRAIN_FILE = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_train_file.csv'
PREDICTIONS_FILENAME = f'{PROPERTY_CODE}_predictions_complete.csv'

# Optional train-set rebalancing by target range
USE_BALANCED_TRAINING = False
BALANCED_TRAIN_FILE = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_train_file_balanced.csv'
BALANCE_TRAINING_BINS = 10
# 'median' caps each OP bin at the median non-empty bin count; use an int for a fixed cap.
BALANCE_TRAINING_MAX_PER_BIN = 'median'

# Chemprop model/training config (preserve OP-specific settings)
BATCH_SIZE = 8
EPOCHS = 100
DEPTH = 3
DROPOUT = 0.025
FFN_HIDDEN_DIM = 50
FFN_NUM_LAYERS = 2
RANDOM_SEED = 11

# Descriptor / transfer settings
USE_FEATURES = True
USE_TRANSFER = False
CHECKPOINT_PATH = Path('')
TRANSFER_FFNN_FROZEN_LAYERS = 2
FEAT_NPZ = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_feat_file.npz'
FEAT_CSV = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_feat_file.csv'

# Evaluation selection (downstream of training)
# None -> use most recently trained model in BASE_OUTPUT_DIR
# int  -> use a specific run, e.g., 30 means ./chemprop_model_output/model30
SELECT_MODEL_NUMBER = 53  # Gao DFT rows removed; see model53/run_manifest.json

# Figure settings
TOP_N_SUBGROUPS = 15
MAX_PER_ROW_SUBGROUPS = 5
PARITY_TRAIN_DENSITY_SCALE = 0.1
MAX_HEAVY_ATOMS_HIST = 100
HEAVY_ATOM_HIST_FIGSIZE = (3.5, 1.4)
BEST_WORST_HETEROATOM_CARBON_RATIO_BINS = 16
BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS = True
BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER = 1.5

# Error diagnostics settings
ERROR_REVIEW_SPLIT = 'test'
TOP_N_BEST_POINTS = 25
TOP_N_WORST_POINTS = 25
TOP_N_WORST_POINTS_ALL_SETS = 15
ERROR_DIAGNOSTICS_PREVIEW_ROWS = 12
SHOW_ALLSETS_WORST_GRID = False
MAX_WORST_GRID_MOLECULES = 12
SHOW_OUTLIER_TABLE = False
TOP_N_REVIEW_SUBGROUP_MOLECULES = 50
SUBGROUP_TTEST_MIN_SELECTED_COUNT = 2
SUBGROUP_TTEST_Q_THRESHOLD = 0.05
OUTLIER_Z_THRESHOLD = 3.0
OUTLIER_IQR_MULTIPLIER = 1.5
EXPORT_ERROR_REVIEW = True

print(f'Train file: {TRAIN_FILE}')
print(f'Output root: {BASE_OUTPUT_DIR}')


## 2. Train/Predict Helpers


In [ ]:
def ensure_cli_available() -> str:
    # Jupyter kernels do not necessarily inherit the shell's Conda PATH.
    python_dir = Path(sys.executable).resolve().parent
    candidates = [python_dir / 'Scripts' / 'chemprop.exe',
                  python_dir / 'chemprop.exe', python_dir / 'chemprop']
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise EnvironmentError(
        f'Chemprop CLI is not installed for notebook Python {sys.executable}. '
        'Select the Python (chemprop2) kernel and restart the notebook.'
    )


def _next_model_dir(base_dir: Path) -> Path:
    base_dir.mkdir(parents=True, exist_ok=True)
    existing = [
        d for d in base_dir.iterdir()
        if d.is_dir() and re.match(r'model\d+', d.name)
    ]
    last_num = max((int(re.search(r'\d+', d.name).group()) for d in existing), default=0)
    return base_dir / f'model{last_num + 1}'


def _latest_model_dir(base_dir: Path) -> Path | None:
    if not base_dir.exists():
        return None
    candidates = [
        d for d in base_dir.iterdir()
        if d.is_dir() and re.match(r'model\d+', d.name)
    ]
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: int(re.search(r'\d+', p.name).group()))[-1]


def _run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print('Running:', ' '.join(map(str, cmd)))
    completed = subprocess.run(
        cmd,
        check=False,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise subprocess.CalledProcessError(
            completed.returncode,
            cmd,
            output=completed.stdout,
            stderr=completed.stderr,
        )


def _preflight_checks(
    train_file: Path,
    smiles_col: str,
    target_col: str,
    split_col: str,
    use_features: bool,
    feat_npz: Path,
    feat_csv: Path,
    use_transfer: bool,
    checkpoint_path: Path,
) -> None:
    ensure_cli_available()

    if not train_file.exists():
        raise FileNotFoundError(f'Training file not found: {train_file}')

    train_df = pd.read_csv(train_file)
    required_cols = {smiles_col, target_col, split_col}
    missing = required_cols - set(train_df.columns)
    if missing:
        raise ValueError(f'Training file missing required columns: {sorted(missing)}')

    if use_features:
        if not feat_npz.exists():
            raise FileNotFoundError(f'Descriptor NPZ not found: {feat_npz}')
        if not feat_csv.exists():
            raise FileNotFoundError(f'Descriptor CSV not found: {feat_csv}')

    if use_transfer and not checkpoint_path.exists():
        raise FileNotFoundError(f'Transfer checkpoint not found: {checkpoint_path}')


def make_range_balanced_training_file(
    source_file: Path,
    output_file: Path,
    target_col: str,
    split_col: str = 'split',
    train_split: str = 'train',
    n_bins: int = 10,
    max_per_bin: int | str | None = 'median',
    seed: int = 42,
) -> Path:
    source_df = pd.read_csv(source_file)
    required_cols = {target_col, split_col}
    missing = required_cols - set(source_df.columns)
    if missing:
        raise ValueError(f'Training file missing required columns: {sorted(missing)}')

    train_rows = source_df[source_df[split_col] == train_split].copy()
    holdout_rows = source_df[source_df[split_col] != train_split].copy()
    if train_rows.empty:
        raise ValueError(f'No rows found with {split_col} == {train_split!r}')

    target_values = pd.to_numeric(train_rows[target_col], errors='coerce')
    if target_values.isna().any():
        raise ValueError(f'{target_col} contains missing or non-numeric values in the training split')
    if target_values.nunique() < 2:
        raise ValueError(f'{target_col} needs at least two unique training values for range balancing')

    bin_edges = np.linspace(target_values.min(), target_values.max(), n_bins + 1)
    train_rows = train_rows.assign(
        _target_bin=pd.cut(target_values, bins=bin_edges, include_lowest=True)
    )
    bin_counts_before = train_rows['_target_bin'].value_counts(sort=False)
    nonempty_counts = bin_counts_before[bin_counts_before > 0]

    if isinstance(max_per_bin, str):
        if max_per_bin == 'median':
            cap = int(np.ceil(nonempty_counts.median()))
        elif max_per_bin == 'min_nonempty':
            cap = int(nonempty_counts.min())
        else:
            raise ValueError("max_per_bin must be 'median', 'min_nonempty', an int, or None")
    elif max_per_bin is None:
        cap = int(nonempty_counts.max())
    else:
        cap = int(max_per_bin)
    if cap < 1:
        raise ValueError('max_per_bin must resolve to at least 1 row per non-empty bin')

    balanced_train_with_bins = (
        train_rows.groupby('_target_bin', observed=True, group_keys=False)
        .apply(lambda group: group.sample(n=min(len(group), cap), random_state=seed))
    )
    balanced_train = (
        balanced_train_with_bins.drop(columns='_target_bin')
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )
    balanced_df = pd.concat([balanced_train, holdout_rows], ignore_index=True)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    balanced_df.to_csv(output_file, index=False)

    bin_counts_after = (
        balanced_train_with_bins['_target_bin']
        .value_counts(sort=False)
        .reindex(bin_counts_before.index, fill_value=0)
    )
    balance_summary_df = pd.DataFrame({
        'before': bin_counts_before.astype(int),
        'after': bin_counts_after.astype(int),
    })
    balance_summary_df.index = balance_summary_df.index.astype(str)

    print(f'Balanced training file saved to: {output_file}')
    print(f'Training rows before/after: {len(train_rows)} -> {len(balanced_train)}')
    print(f'Rows retained per non-empty {target_col} bin: up to {cap}')
    display(balance_summary_df)
    print('Split counts:', balanced_df[split_col].value_counts().to_dict())

    return output_file


def train_and_predict(
    property_code: str,
    target_col: str,
    smiles_col: str = 'SMILES',
    split_col: str = 'split',
    base_dir: Path = Path('chemprop_model_output'),
    train_file: Path = Path('./Datasets/trainable_data/property_chemprop_train_file.csv'),
    use_features: bool = False,
    feat_npz: Path | None = None,
    feat_csv: Path | None = None,
    use_transfer: bool = False,
    checkpoint_path: Path | None = None,
    transfer_frozen_layers: int = 2,
    run_train: bool = True,
    run_predict: bool = True,
    epochs: int = 100,
    batch_size: int = 8,
    depth: int = 3,
    dropout: float = 0.025,
    ffn_hidden_dim: int = 300,
    ffn_num_layers: int = 1,
    seed: int = 42,
    ) -> dict:
    feat_npz = feat_npz or Path('')
    feat_csv = feat_csv or Path('')
    checkpoint_path = checkpoint_path or Path('')

    _preflight_checks(
        train_file=train_file,
        smiles_col=smiles_col,
        target_col=target_col,
        split_col=split_col,
        use_features=use_features,
        feat_npz=feat_npz,
        feat_csv=feat_csv,
        use_transfer=use_transfer,
        checkpoint_path=checkpoint_path,
    )

    if run_train:
        model_dir = _next_model_dir(base_dir)
        model_dir.mkdir(parents=True, exist_ok=True)
    else:
        latest = _latest_model_dir(base_dir)
        if latest is None:
            raise FileNotFoundError(f'No existing model directories found in {base_dir}')
        model_dir = latest

    print(f'Using model directory: {model_dir}')

    train_cmd = [
        ensure_cli_available(), 'train',
        '--data-path', str(train_file),
        '--task-type', 'regression',
        '--output-dir', str(model_dir),
        '--batch-size', str(batch_size),
        '--epochs', str(epochs),
        '--depth', str(depth),
        '--dropout', str(dropout),
        '--ffn-hidden-dim', str(ffn_hidden_dim),
        '--ffn-num-layers', str(ffn_num_layers),
        '--pytorch-seed', str(seed),
    ]

    train_cmd.extend(['--splits-column', split_col])

    if use_features:
        train_cmd.extend(['--descriptors-path', str(feat_npz)])

    if use_transfer:
        train_cmd.extend([
            '--checkpoint', str(checkpoint_path),
            '--freeze-encoder',
            '--frzn-ffn-layers', str(transfer_frozen_layers),
        ])

    if run_train:
        _run_command(train_cmd)

    preds_dir = model_dir / 'predictions'
    preds_dir.mkdir(parents=True, exist_ok=True)

    preds_file = preds_dir / 'set_preds.csv'
    merged_output_file = model_dir / f'{property_code}_predictions_complete.csv'

    predict_cmd = [
        ensure_cli_available(), 'predict',
        '--test-path', str(train_file),
        '--model-paths', str(model_dir),
        '--preds-path', str(preds_file),
    ]

    if use_features:
        predict_cmd.extend(['--descriptors-path', str(feat_npz)])

    if run_predict:
        _run_command(predict_cmd)

    if not preds_file.exists():
        raise FileNotFoundError(f'Predictions file not found: {preds_file}')

    preds_df = pd.read_csv(preds_file)
    train_df = pd.read_csv(train_file)

    complete_data = {
        smiles_col: train_df[smiles_col],
        f'{property_code}_true': train_df[target_col],
        f'{property_code}_pred': preds_df.iloc[:, 1],
        split_col: train_df[split_col],
    }

    if use_features:
        feat_df = pd.read_csv(feat_csv)
        if 'T_C' in feat_df.columns:
            complete_data['T_C'] = feat_df['T_C']

    complete_df = pd.DataFrame(complete_data)
    complete_df.to_csv(merged_output_file, index=False)

    manifest = {
        'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
        'property_code': property_code,
        'target_column': target_col,
        'smiles_column': smiles_col,
        'split_column': split_col,
        'train_file': str(train_file),
        'model_dir': str(model_dir),
        'preds_file': str(preds_file),
        'merged_output_file': str(merged_output_file),
        'use_features': use_features,
        'use_transfer': use_transfer,
        'checkpoint_path': str(checkpoint_path) if use_transfer else None,
        'transfer_frozen_layers': transfer_frozen_layers if use_transfer else None,
        'epochs': epochs,
        'batch_size': batch_size,
        'depth': depth,
        'dropout': dropout,
        'ffn_hidden_dim': ffn_hidden_dim,
        'ffn_num_layers': ffn_num_layers,
        'seed': seed,
        'train_cmd': train_cmd,
        'predict_cmd': predict_cmd,
    }

    manifest_path = model_dir / 'run_manifest.json'
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

    legacy_output_file = model_dir / PREDICTIONS_FILENAME
    if merged_output_file.name != legacy_output_file.name:
        complete_df.to_csv(legacy_output_file, index=False)

    artifacts = {
        'model_dir': model_dir,
        'preds_file': preds_file,
        'merged_output_file': merged_output_file,
        'legacy_output_file': legacy_output_file,
        'manifest_path': manifest_path,
        'metrics_csv': model_dir / 'model_0' / 'trainer_logs' / 'version_0' / 'metrics.csv',
        'best_pt': model_dir / 'model_0' / 'best.pt',
    }

    print(f'Merged predictions saved to: {merged_output_file}')
    print(f'Run manifest saved to: {manifest_path}')

    return artifacts


## 3. Run Train/Predict


In [ ]:
ACTIVE_TRAIN_FILE = TRAIN_FILE
if USE_BALANCED_TRAINING:
    if USE_FEATURES:
        raise NotImplementedError('Balanced training also needs matching downsampled descriptor files when USE_FEATURES=True.')
    ACTIVE_TRAIN_FILE = make_range_balanced_training_file(
        source_file=TRAIN_FILE,
        output_file=BALANCED_TRAIN_FILE,
        target_col=TARGET_COLUMN,
        split_col=SPLIT_COLUMN,
        n_bins=BALANCE_TRAINING_BINS,
        max_per_bin=BALANCE_TRAINING_MAX_PER_BIN,
        seed=RANDOM_SEED,
    )
else:
    print(f'Using original training file: {ACTIVE_TRAIN_FILE}')

print(f'Using split column for training: {SPLIT_COLUMN}')

run_artifacts = None
if RUN_TRAIN or RUN_PREDICT:
    run_artifacts = train_and_predict(
        property_code=PROPERTY_CODE,
        target_col=TARGET_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_col=SPLIT_COLUMN,
        base_dir=BASE_OUTPUT_DIR,
        train_file=ACTIVE_TRAIN_FILE,
        use_features=USE_FEATURES,
        feat_npz=FEAT_NPZ,
        feat_csv=FEAT_CSV,
        use_transfer=USE_TRANSFER,
        checkpoint_path=CHECKPOINT_PATH,
        transfer_frozen_layers=TRANSFER_FFNN_FROZEN_LAYERS,
        run_train=RUN_TRAIN,
        run_predict=RUN_PREDICT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        depth=DEPTH,
        dropout=DROPOUT,
        ffn_hidden_dim=FFN_HIDDEN_DIM,
        ffn_num_layers=FFN_NUM_LAYERS,
        seed=RANDOM_SEED,
    )


def build_artifacts_for_model(model_dir: Path, property_code: str) -> dict:
    return {
        'model_dir': model_dir,
        'preds_file': model_dir / 'predictions' / 'set_preds.csv',
        'merged_output_file': model_dir / f'{property_code}_predictions_complete.csv',
        'legacy_output_file': model_dir / PREDICTIONS_FILENAME,
        'manifest_path': model_dir / 'run_manifest.json',
        'metrics_csv': model_dir / 'model_0' / 'trainer_logs' / 'version_0' / 'metrics.csv',
        'best_pt': model_dir / 'model_0' / 'best.pt',
    }


if SELECT_MODEL_NUMBER is None:
    if run_artifacts is not None:
        selected_model_dir = Path(run_artifacts['model_dir'])
    else:
        latest = _latest_model_dir(BASE_OUTPUT_DIR)
        if latest is None:
            raise FileNotFoundError(f'No existing model directories found in {BASE_OUTPUT_DIR}')
        selected_model_dir = latest
    selected_label = f'latest ({selected_model_dir.name})'
else:
    selected_model_dir = BASE_OUTPUT_DIR / f'model{int(SELECT_MODEL_NUMBER)}'
    if not selected_model_dir.exists():
        raise FileNotFoundError(f'Selected model directory not found: {selected_model_dir}')
    selected_label = f'model{int(SELECT_MODEL_NUMBER)}'

artifacts = build_artifacts_for_model(selected_model_dir, PROPERTY_CODE)

run_predictions_path = artifacts['merged_output_file']
if not run_predictions_path.exists() and artifacts['legacy_output_file'].exists():
    run_predictions_path = artifacts['legacy_output_file']

if not run_predictions_path.exists():
    raise FileNotFoundError(
        f'Predictions CSV not found for selected run ({selected_label}). '
        f'Looked for: {artifacts["merged_output_file"]} and {artifacts["legacy_output_file"]}'
    )

print(f'Downstream model selection: {selected_label}')
print('Current run artifacts used downstream:')
for k, v in artifacts.items():
    print(f'- {k}: {v}')


## 4. Evaluation Metrics


In [ ]:
metrics_df = pd.read_csv(run_predictions_path)

true_col = f'{PROPERTY_CODE}_true'
pred_col = f'{PROPERTY_CODE}_pred'

required_cols = {true_col, pred_col, SPLIT_COLUMN}
missing = required_cols - set(metrics_df.columns)
if missing:
    raise ValueError(f'Missing required columns for MAE calculation: {sorted(missing)}')

if 'summarize_target_distribution' in globals():
    target_stats_df = summarize_target_distribution(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
    )
else:
    stats_frames = []
    overall_stats_df = (
        metrics_df[true_col]
        .agg(['min', 'max', 'mean', 'median'])
        .to_frame()
        .T.assign(split='overall')
    )
    stats_frames.append(overall_stats_df)
    split_stats_df = (
        metrics_df.groupby(SPLIT_COLUMN, dropna=False)[true_col]
        .agg(['min', 'max', 'mean', 'median'])
        .reset_index()
        .rename(columns={SPLIT_COLUMN: 'split'})
    )
    stats_frames.append(split_stats_df)
    target_stats_df = pd.concat(stats_frames, ignore_index=True)[['split', 'min', 'max', 'mean', 'median']]
print('Target distribution summary:')
display(target_stats_df)

for split_name in ['train', 'val', 'test']:
    split_df = metrics_df[metrics_df[SPLIT_COLUMN] == split_name].copy()
    if split_df.empty:
        print(f'{split_name.capitalize()} MAE: split not found in {run_predictions_path}')
        continue
    mae = (split_df[true_col] - split_df[pred_col]).abs().mean()
    print(f'{split_name.capitalize()} MAE: {mae:.4f} (n={len(split_df)})')


## 5. Figure Helpers


In [ ]:
import importlib
import sys
EXTERNAL_FUNCTIONS_DIR = (Path.cwd().parent / 'External Functions').resolve()
PLOTTING_FUNCTIONS_DIR = (EXTERNAL_FUNCTIONS_DIR / 'Plotting Functions').resolve()
for path in (EXTERNAL_FUNCTIONS_DIR, PLOTTING_FUNCTIONS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import train_prediction_plotting as _train_prediction_plotting
importlib.reload(_train_prediction_plotting)

from train_prediction_plotting import (
    build_prediction_error_table,
    compare_ranked_subgroups_to_training_for_run,
    plot_atom_count_hist,
    plot_best_worst_heteroatom_carbon_ratio_for_run,
    plot_best_worst_molecules_for_run,
    plot_heteroatom_pie,
    plot_loss_curves,
    plot_parity_scatter_for_run,
    plot_parity_scatter_grid_for_run,
    plot_stacked_test_parity_for_runs,
    plot_ranked_subgroups_for_run,
    plot_subgroups_for_run,
    plot_training_density_for_run,
    summarize_target_distribution,
)


## 6. Generate Figures


In [ ]:
if RUN_FIGURES:
    figures_dir = artifacts['model_dir'] / 'figures'
    figures_dir.mkdir(parents=True, exist_ok=True)

    plot_loss_curves(artifacts['metrics_csv'], save_path=figures_dir / 'loss_curves.png')

    plot_training_density_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        density_split='train',
        density_scale=PARITY_TRAIN_DENSITY_SCALE,
        save_path=figures_dir / f'{PROPERTY_CODE}_train_density.png',
    )

    plot_parity_scatter_grid_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        split_names=('train', 'val', 'test'),
        density_split='train',
        x_axis_range=(-1.5, 3),
        y_axis_range=(-1.5, 3),
        x_tick_values=[-1, 0, 1, 2, 3],
        y_tick_values=[-1, 0, 1, 2, 3],
        save_path=figures_dir / f'parity_plot_grid.png',
    )

    plot_subgroups_for_run(
        run_predictions_path,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name='train',
        top_n=TOP_N_SUBGROUPS,
        max_per_row=MAX_PER_ROW_SUBGROUPS,
        outfile=figures_dir / 'train_top_subgroups.png',
        title='Top Molecular Subgroups: Training Set',
    )

    best_worst_summary = plot_best_worst_molecules_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        top_n=max(TOP_N_BEST_POINTS, TOP_N_WORST_POINTS),
        save_dir=figures_dir,
    )

    best_subgroup_summary = plot_ranked_subgroups_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        ranking='best',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        top_n_subgroups=TOP_N_SUBGROUPS,
        max_per_row=MAX_PER_ROW_SUBGROUPS,
        outfile=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_subgroups.png',
    )
    worst_subgroup_summary = plot_ranked_subgroups_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        ranking='worst',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        top_n_subgroups=TOP_N_SUBGROUPS,
        max_per_row=MAX_PER_ROW_SUBGROUPS,
        outfile=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_worst_subgroups.png',
    )

    best_subgroup_ttest_df = compare_ranked_subgroups_to_training_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        ranking='best',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        baseline_split='train',
        min_selected_count=SUBGROUP_TTEST_MIN_SELECTED_COUNT,
    )
    worst_subgroup_ttest_df = compare_ranked_subgroups_to_training_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        ranking='worst',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        baseline_split='train',
        min_selected_count=SUBGROUP_TTEST_MIN_SELECTED_COUNT,
    )

    best_subgroup_ttest_path = artifacts['model_dir'] / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_subgroup_ttest.csv'
    worst_subgroup_ttest_path = artifacts['model_dir'] / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_worst_subgroup_ttest.csv'
    best_subgroup_ttest_df.to_csv(best_subgroup_ttest_path, index=False)
    worst_subgroup_ttest_df.to_csv(worst_subgroup_ttest_path, index=False)

    subgroup_stats_cols = [
        'fragment_smiles', 'fragment_label', 'selected_count', 'selected_rate',
        'train_count', 'train_rate', 'delta_rate', 'p_value', 'q_value'
    ]
    print('\nBest-subset moieties enriched versus training (Welch t-test on per-molecule presence):')
    display(best_subgroup_ttest_df[best_subgroup_ttest_df['q_value'] <= SUBGROUP_TTEST_Q_THRESHOLD][subgroup_stats_cols].head(20))
    print('\nWorst-subset moieties enriched versus training (Welch t-test on per-molecule presence):')
    display(worst_subgroup_ttest_df[worst_subgroup_ttest_df['q_value'] <= SUBGROUP_TTEST_Q_THRESHOLD][subgroup_stats_cols].head(20))
    print(f'Saved subgroup t-test tables to: {best_subgroup_ttest_path} and {worst_subgroup_ttest_path}')

    plot_atom_count_hist(
        run_predictions_path,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        max_atoms=MAX_HEAVY_ATOMS_HIST,
        hist_bins=45,
        figsize=HEAVY_ATOM_HIST_FIGSIZE,
        save_path=figures_dir / 'atom_count_hist.png',
    )
    plot_heteroatom_pie(
        run_predictions_path,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        save_path=figures_dir / 'heteroatom_pie.png',
    )
    best_worst_heteroatom_carbon_ratio_df = plot_best_worst_heteroatom_carbon_ratio_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        top_n=TOP_N_BEST_POINTS,
        bottom_n=TOP_N_WORST_POINTS,
        bins=BEST_WORST_HETEROATOM_CARBON_RATIO_BINS,
        remove_outliers=BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS,
        outlier_iqr_multiplier=BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER,
        save_path=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_heteroatom_carbon_ratio.png',
    )
    best_worst_heteroatom_carbon_ratio_summary_df = (
        best_worst_heteroatom_carbon_ratio_df
        .dropna(subset=['heteroatom_to_carbon_ratio'])
        .groupby('rank_group', dropna=False)
        .agg(
            n_molecules=('heteroatom_to_carbon_ratio', 'count'),
            n_plotted=('ratio_plotted_in_histogram', 'sum'),
            n_hidden_outliers=('ratio_outlier_for_histogram', 'sum'),
            min=('heteroatom_to_carbon_ratio', 'min'),
            max=('heteroatom_to_carbon_ratio', 'max'),
            mean=('heteroatom_to_carbon_ratio', 'mean'),
            median=('heteroatom_to_carbon_ratio', 'median'),
        )
        .reset_index()
    )
    display(best_worst_heteroatom_carbon_ratio_summary_df)
    best_worst_heteroatom_carbon_ratio_df.to_csv(
        figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_heteroatom_carbon_ratio.csv',
        index=False,
    )

    print(f'Figures saved to: {figures_dir}')
else:
    print('RUN_FIGURES is False; skipping figure generation.')


## 7. Error Diagnostics


In [ ]:
split_df = build_prediction_error_table(
    run_predictions_path,
    property_code=PROPERTY_CODE,
    split_col=SPLIT_COLUMN,
    smiles_col=SMILES_COLUMN,
    split_name=ERROR_REVIEW_SPLIT,
    outlier_z_threshold=OUTLIER_Z_THRESHOLD,
    outlier_iqr_multiplier=OUTLIER_IQR_MULTIPLIER,
)

true_col = f'{PROPERTY_CODE}_true'
pred_col = f'{PROPERTY_CODE}_pred'

iqr_threshold = split_df['abs_error_iqr_threshold'].iloc[0] if len(split_df) else float('nan')
best_points_df = split_df.sort_values('abs_error', ascending=True).head(TOP_N_BEST_POINTS).copy()
worst_points_df = split_df.sort_values('abs_error', ascending=False).head(TOP_N_WORST_POINTS).copy()
outlier_df = split_df[split_df['flag_potential_outlier']].sort_values('abs_error', ascending=False).copy()

display_cols = [
    'best_rank', 'worst_rank', SMILES_COLUMN, true_col, pred_col,
    'residual', 'abs_error', 'residual_z',
    'flag_large_abs_error', 'flag_large_residual_z', 'flag_potential_outlier'
]
all_sets_display_cols = [SPLIT_COLUMN, *display_cols]

def display_preview(df, cols, label, max_rows=ERROR_DIAGNOSTICS_PREVIEW_ROWS):
    shown = min(max_rows, len(df))
    print(f"\n{label} (showing {shown} of {len(df)}):")
    if shown == 0:
        print('  - none')
        return
    display(df[cols].head(shown))

print(f"Error diagnostics split: {ERROR_REVIEW_SPLIT} (n={len(split_df)})")
print(f"IQR threshold (abs error): {iqr_threshold:.4f}")
print(f"Potential outliers flagged: {len(outlier_df)}")

display_preview(
    best_points_df,
    display_cols,
    f"Top {min(TOP_N_BEST_POINTS, len(split_df))} best predictions",
)
display_preview(
    worst_points_df,
    display_cols,
    f"Top {min(TOP_N_WORST_POINTS, len(split_df))} worst predictions",
)

all_sets_df = build_prediction_error_table(
    run_predictions_path,
    property_code=PROPERTY_CODE,
    split_col=SPLIT_COLUMN,
    smiles_col=SMILES_COLUMN,
    split_name=None,
    outlier_z_threshold=OUTLIER_Z_THRESHOLD,
    outlier_iqr_multiplier=OUTLIER_IQR_MULTIPLIER,
)
all_sets_worst_points_df = all_sets_df.sort_values('abs_error', ascending=False).head(TOP_N_WORST_POINTS_ALL_SETS).copy()
display_preview(
    all_sets_worst_points_df,
    all_sets_display_cols,
    f"Top {min(TOP_N_WORST_POINTS_ALL_SETS, len(all_sets_df))} worst predictions across all sets",
)

if SHOW_ALLSETS_WORST_GRID:
    grid_subset_df = all_sets_worst_points_df.head(MAX_WORST_GRID_MOLECULES).copy()
    all_sets_grid_mols = []
    all_sets_grid_legends = []
    all_sets_grid_skipped = 0
    for idx, row in grid_subset_df.reset_index(drop=True).iterrows():
        smiles = str(row[SMILES_COLUMN])
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            all_sets_grid_skipped += 1
            continue
        all_sets_grid_mols.append(mol)
        all_sets_grid_legends.append(
            f"#{idx + 1} {row[SPLIT_COLUMN]} | |err|={row['abs_error']:.3f}\n"
            f"true={row[true_col]:.3f} pred={row[pred_col]:.3f}"
        )

    if all_sets_grid_mols:
        print(f"\nWorst-prediction molecule grid across all sets (showing {len(all_sets_grid_mols)} molecules):")
        display(
            Draw.MolsToGridImage(
                all_sets_grid_mols,
                molsPerRow=4,
                subImgSize=(360, 280),
                legends=all_sets_grid_legends,
            )
        )
        if all_sets_grid_skipped:
            print(f'Skipped {all_sets_grid_skipped} rows with invalid SMILES while building the grid.')
    else:
        print('\nNo valid molecules available to render for the all-sets worst-prediction grid.')
else:
    print('\nSkipping molecule grid (SHOW_ALLSETS_WORST_GRID=False).')

if len(outlier_df) > 0:
    if SHOW_OUTLIER_TABLE:
        display_preview(outlier_df, display_cols, 'Flagged potential outliers')
    else:
        print(f"\nFlagged potential outliers: {len(outlier_df)} (table suppressed; set SHOW_OUTLIER_TABLE=True to display)")
else:
    print('\nNo potential outliers flagged with current thresholds.')

if EXPORT_ERROR_REVIEW:
    review_out_path = Path(artifacts['model_dir']) / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_error_review.csv'
    split_df.sort_values('abs_error', ascending=False).to_csv(review_out_path, index=False)
    all_sets_review_out_path = Path(artifacts['model_dir']) / f'{PROPERTY_CODE}_all_error_review.csv'
    all_sets_df.sort_values('abs_error', ascending=False).to_csv(all_sets_review_out_path, index=False)
    print(f'\nSaved full error review table to: {review_out_path}')
    print(f'Saved all-sets error review table to: {all_sets_review_out_path}')


## 10. Optional Custom SMILES Prediction


In [ ]:
def predict_smiles(smiles_list, model_dir, property_code, use_features=False, feat_npz=None):
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as tmp_file:
        tmp_file_path = tmp_file.name
        pd.DataFrame({'smiles': smiles_list}).to_csv(tmp_file_path, index=False)

    preds_dir = os.path.join(model_dir, 'predictions')
    os.makedirs(preds_dir, exist_ok=True)
    preds_file = os.path.join(preds_dir, 'smiles_preds.csv')

    predict_cmd = [
        ensure_cli_available(), 'predict',
        '--test-path', tmp_file_path,
        '--model-paths', model_dir,
        '--preds-path', preds_file,
    ]

    if use_features and feat_npz is not None:
        predict_cmd.extend(['--descriptors-path', feat_npz])

    subprocess.run(predict_cmd, check=True)

    preds_df = pd.read_csv(preds_file)
    preds_df = preds_df.rename(columns={preds_df.columns[1]: property_code})

    os.remove(tmp_file_path)
    return preds_df


# Generic demonstration input; replace with a molecule of interest.
smiles_list = ['CCO']

# use selected run by default for custom predictions
custom_model_path = str(artifacts['model_dir'] / 'model_0')
preds = predict_smiles(
    smiles_list,
    model_dir=custom_model_path,
    property_code=PROPERTY_CODE,
    use_features=USE_FEATURES,
    feat_npz=str(FEAT_NPZ) if USE_FEATURES else None,
)
print(preds)


## 10. Run Summary


In [ ]:
print('Run summary:')
print(f'- Property code: {PROPERTY_CODE}')
print(f'- Source train file: {TRAIN_FILE}')
print(f'- Active train file: {ACTIVE_TRAIN_FILE}')
print(f'- Balanced training enabled: {USE_BALANCED_TRAINING}')
print(f'- Features enabled: {USE_FEATURES}')
print(f'- Model directory: {artifacts["model_dir"]}')
print(f'- Best model: {artifacts["best_pt"]}')
print(f'- Metrics CSV: {artifacts["metrics_csv"]}')
print(f'- Predictions CSV: {run_predictions_path}')
print(f'- Manifest: {artifacts["manifest_path"]}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CV_COLOR = '#c65a45'
CV_CENTER_V = 0.699
CV_PEAK_SEPARATION_V = 0.060
CV_LINE_WIDTH = 1.4
CV_TARGET_PEAK_CURRENT = 0.75
CV_SWITCH_CURRENT = 0.19
CV_REVERSE_CONNECTOR_CENTER_V = 0.9
CV_REVERSE_CONNECTOR_WIDTH_V = 0.04

# Stylized cyclic voltammogram to illustrate the quinone/hydroquinone redox equivalence.
e_start, e_switch = 0.35, 1.00
forward_scan = np.linspace(e_start, e_switch, 500)
reverse_scan = np.linspace(e_switch, e_start, 500)

oxidation_peak_v = CV_CENTER_V + (CV_PEAK_SEPARATION_V / 2)
reduction_peak_v = CV_CENTER_V - (CV_PEAK_SEPARATION_V / 2)

forward_baseline = -0.010 + 0.010 * (forward_scan - e_start) / (e_switch - e_start)
forward_prewave = 0.075 / (1 + np.exp(-(forward_scan - (CV_CENTER_V - 0.090)) / 0.020))
forward_peak = 0.78 * np.exp(-0.5 * ((forward_scan - oxidation_peak_v) / 0.034) ** 2)
forward_tail_gate = 1 / (1 + np.exp(-(forward_scan - oxidation_peak_v) / 0.018))
forward_tail = 0.62 * np.exp(-(forward_scan - oxidation_peak_v) / 0.140) * forward_tail_gate
pinch_center_v = 0.990
pinch_width_v = 0.028
pinch_current = CV_SWITCH_CURRENT
forward_pinch_gate = 1 / (1 + np.exp((forward_scan - pinch_center_v) / pinch_width_v))
forward_raw = forward_baseline + forward_prewave + forward_peak + forward_tail
forward_current = pinch_current + (forward_raw - pinch_current) * forward_pinch_gate

reverse_progress = (e_switch - reverse_scan)
reverse_shoulder = 0.36 * (1 - np.exp(-reverse_progress / 0.11)) * np.exp(-reverse_progress / 0.34)
reverse_baseline = -0.11 + 0.010 * (reverse_scan - e_start) / (e_switch - e_start)
reverse_peak = 0.92 * np.exp(-0.5 * ((reverse_scan - reduction_peak_v) / 0.044) ** 2)
reverse_tail = 0.16 * np.exp(-0.5 * ((reverse_scan - (reduction_peak_v - 0.12)) / 0.130) ** 2)
reverse_activation = 1 / (1 + np.exp((reverse_scan - CV_REVERSE_CONNECTOR_CENTER_V) / CV_REVERSE_CONNECTOR_WIDTH_V))
reverse_raw = reverse_baseline + reverse_shoulder - reverse_peak - reverse_tail
reverse_current = pinch_current + (reverse_raw - pinch_current) * reverse_activation

# Find the most-positive intersection between the oxidative and reductive sweeps,
# then clip both traces to that shared switching point.
reverse_current_on_forward_grid = reverse_current[::-1]
delta_current = forward_current - reverse_current_on_forward_grid
intersection_indices = np.where(np.sign(delta_current[:-1]) != np.sign(delta_current[1:]))[0]
if len(intersection_indices) > 0:
    intersection_idx = int(intersection_indices[-1])
    x0 = forward_scan[intersection_idx]
    x1 = forward_scan[intersection_idx + 1]
    d0 = delta_current[intersection_idx]
    d1 = delta_current[intersection_idx + 1]
    if np.isclose(d1, d0):
        switching_v = x1
    else:
        switching_v = x0 - d0 * (x1 - x0) / (d1 - d0)
    f0 = forward_current[intersection_idx]
    f1 = forward_current[intersection_idx + 1]
    switching_i = f0 + (switching_v - x0) * (f1 - f0) / (x1 - x0)
else:
    switching_v = e_switch
    switching_i = 0.5 * (forward_current[-1] + reverse_current[0])

forward_mask = forward_scan < switching_v
reverse_mask = reverse_scan < switching_v
forward_plot_v = np.append(forward_scan[forward_mask], switching_v)
forward_plot_i = np.append(forward_current[forward_mask], switching_i)
reverse_plot_v = np.insert(reverse_scan[reverse_mask], 0, switching_v)
reverse_plot_i = np.insert(reverse_current[reverse_mask], 0, switching_i)

peak_current = float(np.max(forward_plot_i)) if len(forward_plot_i) else 1.0
current_scale = 1.0 if peak_current <= 0 else CV_TARGET_PEAK_CURRENT / peak_current
forward_plot_i = forward_plot_i * current_scale
reverse_plot_i = reverse_plot_i * current_scale
switching_i = switching_i * current_scale

# Tuning notes:
# - Shift CV_REVERSE_CONNECTOR_CENTER_V to move where the reverse sweep starts bending toward the pinch.
# - Increase CV_REVERSE_CONNECTOR_WIDTH_V to make that bend smoother and more gradual.
# - Increase CV_SWITCH_CURRENT to raise the natural meeting current on the positive side.
# - Change CV_TARGET_PEAK_CURRENT to rescale the whole voltammogram vertically.

cv_fig, cv_ax = plt.subplots(figsize=(3.5, 3.0), dpi=600)
cv_ax.plot(forward_plot_v, forward_plot_i, color=CV_COLOR, linewidth=CV_LINE_WIDTH)
cv_ax.plot(reverse_plot_v, reverse_plot_i, color=CV_COLOR, linewidth=CV_LINE_WIDTH)
cv_ax.axhline(0.0, color='0.25', linewidth=0.7)
cv_ax.axvline(CV_CENTER_V, color=CV_COLOR, linewidth=0.8, linestyle='--', alpha=0.55)
cv_ax.text(CV_CENTER_V + 0.012, 0.86, r'$E^o = 0.699\,\mathrm{V}$', color=CV_COLOR, fontsize=10, fontname='Arial', fontweight='normal')

cv_ax.set_xlabel('V vs. SHE', fontsize=12, fontname='Arial', fontweight='normal')
cv_ax.set_ylabel('Current', fontsize=12, fontname='Arial', fontweight='normal')
cv_ax.tick_params(axis='both', labelsize=10, width=0.7, length=3.5, direction='out')
for label in [*cv_ax.get_xticklabels(), *cv_ax.get_yticklabels()]:
    label.set_fontname('Arial')
    label.set_fontweight('normal')

cv_ax.spines['top'].set_visible(False)
cv_ax.spines['right'].set_visible(False)
for spine in cv_ax.spines.values():
    spine.set_linewidth(0.7)

cv_ax.set_xlim(e_start, 1.10)
cv_ax.set_ylim(-1.0, 1.0)
cv_ax.margins(x=0.01)
cv_fig.tight_layout()

cv_figures_dir = Path(artifacts['model_dir']) / 'figures'
cv_figures_dir.mkdir(parents=True, exist_ok=True)
cv_save_path = cv_figures_dir / 'synthetic_cyclic_voltammogram_0p699V.png'
cv_fig.savefig(cv_save_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'Saved synthetic cyclic voltammogram to: {cv_save_path}')


In [ ]:
from pathlib import Path

project_root = Path.cwd().parent.resolve()
stacked_parity_save_path = artifacts['model_dir'] / 'figures' / 'test_parity_plots_vp_op_pka_stacked.png'

# Update the VP/pKa model folders below if you want to compare a different saved run.
stacked_parity_specs = [
    {
        'predictions_csv': project_root / '1. Vapor Pressure' / 'chemprop_model_output' / 'model3' / 'VP_predictions_complete.csv',
        'property_code': 'VP',
        'title': '',
        'x_axis_range': (-7, 4),
        'y_axis_range': (-7, 4),
        'x_tick_values': [-6, -4, -2, 0, 2, 4],
        'y_tick_values': [-6, -4, -2, 0, 2, 4],
    },
    {
        'predictions_csv': run_predictions_path,
        'property_code': PROPERTY_CODE,
        'title': '',
        'x_axis_range': (-1.5, 3),
        'y_axis_range': (-1.5, 3),
        'x_tick_values': [-1, 0, 1, 2, 3],
        'y_tick_values': [-1, 0, 1, 2, 3],
    },
    {
        'predictions_csv': project_root / '3. pKa' / 'chemprop_model_output' / 'model4' / 'pka_predictions_complete.csv',
        'property_code': 'pka',
        'title': '',
        'x_axis_range': (0, 18),
        'y_axis_range': (0, 18),
        'x_tick_values': [0, 5, 10, 15],
        'y_tick_values': [0, 5, 10, 15],
    },
]

plot_stacked_test_parity_for_runs(
    stacked_parity_specs,
    save_path=stacked_parity_save_path,
)

print(f'Saved stacked VP/OP/pKa parity figure to: {stacked_parity_save_path}')
